`PART - I : Basic Property Details Web Scraping`

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException,  TimeoutException
import time
import numpy as np
import pandas as pd
import os
import time
import random

In [ ]:
chrome_options = Options()
chrome_options.add_argument("--disable-http2")
chrome_options.add_argument("--incognito")
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_argument("--ignore-certificate-errors")
chrome_options.add_argument("--enable-features=NetworkServiceInProcess")
chrome_options.add_argument("--disable-features=NetworkService")
chrome_options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/93.0.4577.63 Safari/537.36"
)

driver = webdriver.Chrome(options=chrome_options)
url = "https://www.99acres.com/"
driver.get(url)


wait = WebDriverWait(driver, 5)


def wait_for_page_to_load(driver, wait_obj):
    """Wait until the page is fully loaded."""
    title = driver.title
    try:
        wait_obj.until(lambda d: d.execute_script("return document.readyState") == "complete")
    except Exception:
        print(f'The webpage "{title}" did not get fully loaded.\n')
    else:
        print(f'The webpage "{title}" did get fully loaded.\n')

wait_for_page_to_load(driver, wait)

# Search city

try:
    search_bar = wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="keyword2"]')))
except TimeoutException:
    print("Timeout while locating Search Bar.\n")
else:
    search_bar.send_keys("Gurgaon")
    time.sleep(2)

# Serach Bar Loading
try:
    valid_option = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="0"]')))
except TimeoutException:
    print("Timeout while locating valid search option.\n")
else:
    valid_option.click()
    time.sleep(2)

# Click Search

try:
    search_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="searchform_search_btn"]')))
except TimeoutException:
    print('Timeout while clicking on "Search" button.\n')
else:
    search_button.click()
    wait_for_page_to_load(driver, wait)

# Independent House

try:
    indepedent_house = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="2"]/span[2]')))
except Exception:
    print('Timeout while clicking on "indepedent_house" option.\n')
else:
    indepedent_house.click()
    time.sleep(2)

# Click "Verified" filter

try:
    verified = wait.until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                '/html[1]/body[1]/div[1]/div[1]/div[1]/div[4]/div[3]/div[1]/div[3]/section[1]/div[1]/div[1]/div[1]/div[1]/div[1]/div[1]/div[3]/span[2]',
            )
        )
    )
    verified.click()
    time.sleep(5)
except TimeoutException:
    print('Timeout while clicking on "Verified" filter.\n')


all_properties = []  
page_count = 0

while True:
    page_count += 1
    print(f"\n--- PAGE {page_count} ---")
    print("time:", time.strftime("%Y-%m-%d %H:%M:%S"))
    print("session_id:", driver.session_id)

    # Throattling

    if page_count % 3 == 0:
        print("Throttling: sleeping 3s...")
        time.sleep(3)
    if page_count % 5 == 0:
        print("Throttling: sleeping 10s...")
        time.sleep(10)

    
    try:
        time.sleep(5)
        next_page_button = driver.find_element(By.XPATH, "//a[normalize-space()='Next Page >']")
    except NoSuchElementException:
        print(f"\n [COMPLETED] Scraping finished. Total pages navigated: {page_count}.\n")
        break
    else:
        try:
            
            driver.execute_script(
                "window.scrollBy(0, arguments[0].getBoundingClientRect().top - 100);", next_page_button
            )
            time.sleep(5)

            main_container = driver.find_element(By.CSS_SELECTOR, 'div[data-label="SEARCH"]')
            property_containers = main_container.find_elements(By.CLASS_NAME, "tupleNew__contentWrap")
            print(f"Number of Property Found on page {page_count}: {len(property_containers)}")

            for container in property_containers:
                property_data = {}
                try:
                    # property name

                    try:
                        property_data["property_name"] = container.find_element(By.CLASS_NAME, "tupleNew__propType").text
                    except Exception:
                        property_data["property_name"] = np.nan

                    # link 

                    try:
                        property_data["link"] = container.find_element(
                            By.CSS_SELECTOR, ".tupleNew__propertyHeading.ellipsis"
                        ).get_attribute("href")
                    except Exception:
                        property_data["link"] = np.nan

                    # society

                    try:
                        property_data["society"] = container.find_element(By.CLASS_NAME, "tupleNew__locationName").text
                    except Exception:
                        property_data["society"] = np.nan

                    # price

                    try:
                        property_data["price"] = container.find_element(By.CLASS_NAME, "tupleNew__priceValWrap").text
                    except Exception:
                        property_data["price"] = np.nan

                    # area

                    try:
                        property_data["area"] = container.find_element(
                            By.CSS_SELECTOR, "div.tupleNew__perSqftWrap.ellipsis"
                        ).text
                    except Exception:
                        property_data["area"] = np.nan

                    # area with type

                    try:
                        area1 = container.find_element(By.CLASS_NAME, "tupleNew__area1Type").text
                    except Exception:
                        area1 = ""
                    try:
                        area2 = container.find_element(By.CLASS_NAME, "tupleNew__area2Type").text
                    except Exception:
                        area2 = ""
                    try:
                        area_type = container.find_element(By.CLASS_NAME, "tupleNew__areaType").text
                    except Exception:
                        area_type = ""

                    property_data["areawithtype"] = f"{area1} {area2} {area_type}".strip()

                    all_properties.append(property_data)

                except Exception:
                    print("Not able to extract property details!!")

            # move to next page 
            
            next_page_button = wait.until(
                EC.presence_of_element_located((By.XPATH, "//a[normalize-space()='Next Page >']"))
            )
            driver.execute_script("arguments[0].click();", next_page_button)
            print("✔ Next page clicked successfully")
            time.sleep(5)

        except Exception as e:
            print("Timeout or error while clicking on Next Page:", e)
            break

The webpage "India Real Estate Property Site - Buy Sell Rent Properties Portal - 99acres.com" did get fully loaded.

The webpage "Property in Gurgaon - Real Estate in Gurgaon" did get fully loaded.


--- PAGE 1 ---
time: 2025-11-02 10:52:02
session_id: 4c2727f5c2412b34429cae7bbf4c704a
Number of Property Found on page 1: 78
✔ Next page clicked successfully

--- PAGE 2 ---
time: 2025-11-02 10:53:16
session_id: 4c2727f5c2412b34429cae7bbf4c704a
Number of Property Found on page 2: 51
✔ Next page clicked successfully

--- PAGE 3 ---
time: 2025-11-02 10:54:11
session_id: 4c2727f5c2412b34429cae7bbf4c704a
Throttling: sleeping 3s...
Number of Property Found on page 3: 26
✔ Next page clicked successfully

--- PAGE 4 ---
time: 2025-11-02 10:54:43
session_id: 4c2727f5c2412b34429cae7bbf4c704a
Number of Property Found on page 4: 26
✔ Next page clicked successfully

--- PAGE 5 ---
time: 2025-11-02 10:55:13
session_id: 4c2727f5c2412b34429cae7bbf4c704a
Throttling: sleeping 10s...
Number of Property Foun

In [ ]:
independent_house_df = pd.DataFrame(all_properties)

In [ ]:
independent_house_df.to_csv("../..data/web_scraping/indepedent_house_basic_details.csv")

In [ ]:
indepedent_house_basic_details = pd.read_csv("../../data/web_scraping/indepedent_house_basic_details.csv")
indepedent_house_basic_details

,property_name,link,society,price,area
0,"4 Bedroom House in Sector 4, Gurgaon",https://www.99acres.com/4-bhk-bedroom-independ...,"Sector 4, Gurgaon",₹6.6 Cr,"₹14,667 /sqft"
1,"6 Bedroom House in DLF Phase 1, Gurgaon",https://www.99acres.com/6-bhk-bedroom-independ...,North East & Park Facing Corner 18 meter wide ...,₹25 Cr,"₹22,727 /sqft"
2,"2 Bedroom House in Sector 15 Part 2, Gurgaon",https://www.99acres.com/2-bhk-bedroom-independ...,"Sector 15 Part 2, Gurgaon",₹18.99 Cr,"₹42,031 /sqft"
3,"3 Bedroom House in Jhajjar Road, Gurgaon",https://www.99acres.com/3-bhk-bedroom-independ...,"Green Heritage Villas, Jhajjar",₹3.39 Cr,"₹3,766 /sqft"
4,"5 Bedroom House in Sector 109, Gurgaon",https://www.99acres.com/5-bhk-bedroom-independ...,Sobha International City Phase 3,₹14 Cr,"₹38,889 /sqft"
...,...,...,...,...,...
2464,"3 Bedroom House in Sector 49, Gurgaon",https://www.99acres.com/3-bhk-bedroom-independ...,Eros Rosewood City,₹4.5 Cr,"₹32,680 /sqft"
2465,"4 Bedroom House in Pratap Nagar, Gurgaon",https://www.99acres.com/4-bhk-bedroom-independ...,383/13 Partap nagar,₹1.1 Cr,"₹6,077 /sqft"
2466,"4 Bedroom House in DLF Phase 3, Gurgaon",https://www.99acres.com/4-bhk-bedroom-independ...,National Media Centre,₹7 Cr,"₹32,407 /sqft"
2467,"2 Bedroom House in Devilal Colony, Gurgaon",https://www.99acres.com/2-bhk-bedroom-independ...,"Devilal Colony, Gurgaon",₹70 Lac,"₹76,923 /sqft"


In [27]:
indepedent_house_basic_details['property_id']  = indepedent_house_basic_details['link'].str.split("spid-").str.get(1)
indepedent_house_basic_details

,property_name,link,society,price,area,property_id
0,"4 Bedroom House in Sector 4, Gurgaon",https://www.99acres.com/4-bhk-bedroom-independ...,"Sector 4, Gurgaon",₹6.6 Cr,"₹14,667 /sqft",N80213331
1,"6 Bedroom House in DLF Phase 1, Gurgaon",https://www.99acres.com/6-bhk-bedroom-independ...,North East & Park Facing Corner 18 meter wide ...,₹25 Cr,"₹22,727 /sqft",Q84155382
2,"2 Bedroom House in Sector 15 Part 2, Gurgaon",https://www.99acres.com/2-bhk-bedroom-independ...,"Sector 15 Part 2, Gurgaon",₹18.99 Cr,"₹42,031 /sqft",S82406206
3,"3 Bedroom House in Jhajjar Road, Gurgaon",https://www.99acres.com/3-bhk-bedroom-independ...,"Green Heritage Villas, Jhajjar",₹3.39 Cr,"₹3,766 /sqft",J84873064
4,"5 Bedroom House in Sector 109, Gurgaon",https://www.99acres.com/5-bhk-bedroom-independ...,Sobha International City Phase 3,₹14 Cr,"₹38,889 /sqft",Q83769064
...,...,...,...,...,...,...
2464,"3 Bedroom House in Sector 49, Gurgaon",https://www.99acres.com/3-bhk-bedroom-independ...,Eros Rosewood City,₹4.5 Cr,"₹32,680 /sqft",V81604971
2465,"4 Bedroom House in Pratap Nagar, Gurgaon",https://www.99acres.com/4-bhk-bedroom-independ...,383/13 Partap nagar,₹1.1 Cr,"₹6,077 /sqft",Q81598269
2466,"4 Bedroom House in DLF Phase 3, Gurgaon",https://www.99acres.com/4-bhk-bedroom-independ...,National Media Centre,₹7 Cr,"₹32,407 /sqft",Q81594211
2467,"2 Bedroom House in Devilal Colony, Gurgaon",https://www.99acres.com/2-bhk-bedroom-independ...,"Devilal Colony, Gurgaon",₹70 Lac,"₹76,923 /sqft",D81585663


In [28]:
indepedent_house_basic_details.isnull().sum()

property_name    0
link             0
society          0
price            0
area             6
property_id      0
dtype: int64

In [29]:
indepedent_house_basic_details.duplicated().sum()

0

` PART - II : Detailed Property Page Extractions `

In [ ]:
# Constants

OUTPUT_FILE  = "../../data/web_scraping/independent_house_detailed_page2.csv"
BATCH_SIZE   = 50
BATCH_SLEEP  = 30   # seconds between batches
MIN_DELAY    = 2    # min seconds between individual requests
MAX_DELAY    = 5    # max seconds between individual requests


# Browser Setup

def create_driver() -> webdriver.Chrome:
    """Sets up a headless Chrome browser with human-like headers."""
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    )
    return webdriver.Chrome(options=options)


# Page Load Helper

def wait_for_page_load(driver, wait):
    """Waits until the browser signals the page is fully loaded."""
    try:
        wait.until(lambda d: d.execute_script("return document.readyState") == "complete")
    except TimeoutException:
        print(f"Timeout loading: {driver.title}")


# Field Extractors

def extract_basic_details(driver):
    """
    Extracts structured fields from the property details table:
    bedrooms, bathrooms, balcony, additional rooms, floor, facing, property age, area.
    """
    data = {
        "areawithtype"   : np.nan,
        "bedrooms"       : np.nan,
        "bathrooms"      : np.nan,
        "balcony"        : np.nan,
        "additional_room": np.nan,
        "floor_info"     : np.nan,
        "facing"         : np.nan,
        "property_age"   : np.nan,
        "property_id"    : np.nan
    }

    rows = driver.find_elements(
        By.CSS_SELECTOR,
        "table[data-label='BASIC_DETAILS'] tbody tr"
    )

    for row in rows:
        headers = row.find_elements(By.CLASS_NAME, "component__tableHead")
        for header in headers:
            label = header.text

            if "Area" in label:
                try:
                    data["areawithtype"] = row.find_element(
                        By.CSS_SELECTOR, "div.component__details2"
                    ).text.strip()
                except: pass

            elif "Configuration" in label:
                try:
                    data["bedrooms"]  = row.find_element(By.ID, "bedRoomNum").text.strip()
                    data["bathrooms"] = row.find_element(By.ID, "bathroomNum").text.strip()
                    data["balcony"]   = row.find_element(By.ID, "balconyNum").text.strip()
                except: pass
                try:
                    data["additional_room"] = row.find_element(By.ID, "additionalRooms").text.strip()
                except NoSuchElementException: pass

            elif "Floor Number" in label:
                try:
                    data["floor_info"] = row.find_element(By.ID, "floorNumLabel").text.strip()
                except: pass

            elif "Facing" in label:
                try:
                    data["facing"] = row.find_element(By.ID, "facingLabel").text.strip()
                except: pass

            elif "Property Age" in label:
                try:
                    data["property_age"] = row.find_element(By.ID, "agePossessionLbl").text.strip()
                except: pass

    try:
        data["property_id"] = driver.find_element(
            By.CSS_SELECTOR, "span#Prop_Id"
        ).text.strip()
    except: pass

    return data


def extract_furnishing_details(driver):
    """Extracts list of furnishing items present in the property."""
    try:
        container = driver.find_element(By.ID, "FurnishDetails")
        items = container.find_elements(By.CSS_SELECTOR, "ul#features li div")
        return [item.text.strip() for item in items if item.text.strip()]
    except:
        return np.nan


def extract_amenities(driver):
    """Extracts list of society/property amenities."""
    try:
        container = driver.find_element(By.CSS_SELECTOR, "div[data-label='FACILITIES']")
        items = container.find_elements(By.CSS_SELECTOR, "ul#features li div")
        return [item.text.strip() for item in items if item.text.strip()]
    except:
        return np.nan


def extract_nearby_locations(driver):
    """
    Extracts nearby landmarks by paginating through the carousel.
    Clicks the right arrow until no more new locations appear.
    """
    try:
        container     = driver.find_element(By.ID, "LANDMARK")
        location_elems = container.find_elements(
            By.CSS_SELECTOR, ".NearByLocation__tagWrap .NearByLocation__infoText"
        )
        locations = set(e.text.strip() for e in location_elems if e.text.strip())

        while True:
            try:
                arrow = container.find_element(
                    By.CSS_SELECTOR,
                    ".carousel__arrowContainerBox.carousel__right .carousel__rightArrow"
                )
                driver.execute_script("arguments[0].click();", arrow)
                time.sleep(2)
                container      = driver.find_element(By.ID, "LANDMARK")
                location_elems = container.find_elements(
                    By.CSS_SELECTOR, ".NearByLocation__tagWrap .NearByLocation__infoText"
                )
                locations.update(e.text.strip() for e in location_elems if e.text.strip())
            except:
                break

        return list(locations)
    except:
        return []


# Main Scraper — Single Property Page

def scrape_property(driver, wait, link):
    """
    Visits a single property page and extracts all available fields.
    Returns a flat dictionary ready to be appended to a DataFrame.
    """
    driver.get(link)
    wait_for_page_load(driver, wait)

    data = {"link": link}
    data.update(extract_basic_details(driver))
    data["furnishing_details"] = extract_furnishing_details(driver)
    data["features"]           = extract_amenities(driver)
    data["nearby_location"]    = extract_nearby_locations(driver)


    return data


# Batch Processing Pipeline

def load_existing_data(output_file):
    """Loads already-processed data so we never re-scrape a link."""
    if os.path.exists(output_file):
        existing_df     = pd.read_csv(output_file)
        processed_links = set(existing_df["link"].tolist())
        print(f"Resuming — {len(processed_links)} links already processed.")
    else:
        existing_df     = pd.DataFrame()
        processed_links = set()
    return existing_df, processed_links


def save_batch(batch, output_file):
    """Appends a completed batch to the output CSV."""
    new_df = pd.DataFrame(batch)

    if os.path.exists(output_file):
        existing_df  = pd.read_csv(output_file)
        combined_df  = pd.concat([existing_df, new_df], ignore_index=True)
        combined_df  = combined_df.drop_duplicates(subset=["link"], keep="last")
    else:
        combined_df = new_df

    combined_df.to_csv(output_file, index=False)
    print(f"💾 Batch saved — {len(combined_df)} total records in {output_file}")


def run_scraper(links_file, start_from):
    """
    Main pipeline — processes all links in batches of BATCH_SIZE.
    Saves after every batch so progress is never lost.
    start_from: 1-based index to resume from after an interruption.
    """
    df              = pd.read_csv(links_file)
    links           = df["link"].tolist()
    total           = len(links)
    start           = start_from - 1  # convert to 0-based index

    _, processed_links = load_existing_data(OUTPUT_FILE)

    driver = create_driver()
    wait   = WebDriverWait(driver, 15)

    try:
        while start < total:
            end   = min(start + BATCH_SIZE, total)
            batch = []

            print(f"\n🔄 Processing batch {start + 1} → {end} of {total} properties")


            for i in range(start, end):
                link = links[i]

                if link in processed_links:
                    print(f"⏭️  Skipping (already done): {link}")
                    continue

                data = scrape_property(driver, wait, link)
                batch.append(data)
                time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

            if batch:
                save_batch(batch, OUTPUT_FILE)
                processed_links.update(d["link"] for d in batch)

            start = end

            if end < total:
                print(f"⏳ Sleeping {BATCH_SLEEP}s before next batch...")
                time.sleep(BATCH_SLEEP)

        print("\n✅ All links processed successfully.")

    except Exception as e:
        print(f"\n❌ Interrupted at link {start + 1}: {e}")
        if batch:
            save_batch(batch, OUTPUT_FILE)
            print("💾 Partial batch saved before exit.")

    finally:
        driver.quit()
        print("🔒 Browser closed.")


# Entry Point

if __name__ == "__main__":
    start_from = int(input("Enter link number to start from (1 = beginning): "))
    run_scraper(
        links_file = "../../data/web_scraping/indepedent_house_basic_details.csv",
        start_from = start_from
    )

In [ ]:
indepedent_house_detailed_page = pd.read_csv("../../data/web_scraping/indepedent_house_detailed_page.csv")
indepedent_house_detailed_page

,link,bedrooms,bathrooms,balcony,additional_room,floor_info,facing,property_age,property_id,areawithtype,furnishing_details,features,nearby_location
0,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,2 Balconies,Study Room,2 Floors,South,10+ Year Old,N80213331,4500 sqft (418.06 sqm) Plot Area,"['5 Fan', '10 Light', '1 Chimney', '1 Modular ...","['Park', 'Rain Water Harvesting']","['Palam Vihar Vyapar kendra', 'Palam vihar rai..."
1,https://www.99acres.com/6-bhk-bedroom-independ...,6 Bedrooms,10 Bathrooms,3+ Balconies,"Pooja Room,Study Room,Servant Room,Store Room",4 Floors,North-East,1 to 5 Year Old,Q84155382,11000 sqft (1021.93 sqm) Plot Area,"['1 Water Purifier', '13 Fan', '1 Exhaust Fan'...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...","['Anahat Hospital', 'Sikandarpur rmrg metro st..."
2,https://www.99acres.com/2-bhk-bedroom-independ...,2 Bedrooms,2 Bathrooms,No Balcony,NaN,1 Floors,NaN,1 to 5 Year Old,Y84995768,650 sqft (60.39 sqm) Carpet Area,NaN,NaN,"['Raheja Mall', 'Vipul Trade Centre', 'JMD Meg..."
3,https://www.99acres.com/2-bhk-bedroom-independ...,2 Bedrooms,2 Bathrooms,No Balcony,NaN,NaN,NaN,NaN,S82406206,4518 sqft (419.74 sqm) Plot Area,[],NaN,"['govt sec school', 'Ahmed Hospital Multi Spec..."
4,https://www.99acres.com/3-bhk-bedroom-independ...,3 Bedrooms,4 Bathrooms,3 Balconies,"Study Room,Servant Room,Store Room",2 Floors,South,1 to 5 Year Old,J84873064,9000 sqft (836.13 sqm) Plot Area,"['1 Water Purifier', '10 Fan', '1 Fridge', '1 ...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...",[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2461,https://www.99acres.com/3-bhk-bedroom-independ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[]
2462,https://www.99acres.com/9-bhk-bedroom-independ...,9 Bedrooms,6 Bathrooms,3 Balconies,"Pooja Room,Study Room,Servant Room,Others",3 Floors,North,1 to 5 Year Old,J82367326,900 sqft (83.61 sqm) Plot Area,"['1 Water Purifier', '9 Fan', '1 Microwave', '...","['Feng Shui / Vaastu Compliant', 'Water Storag...",[]
2463,https://www.99acres.com/2-bhk-bedroom-independ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[]
2464,https://www.99acres.com/3-bhk-bedroom-independ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[]


In [31]:
indepedent_house_merge = indepedent_house_basic_details.merge(indepedent_house_detailed_page, on='property_id',how= 'inner')
indepedent_house_merge

,property_name,link_x,society,price,area,property_id,link_y,bedrooms,bathrooms,balcony,additional_room,floor_info,facing,property_age,areawithtype,furnishing_details,features,nearby_location
0,"4 Bedroom House in Sector 4, Gurgaon",https://www.99acres.com/4-bhk-bedroom-independ...,"Sector 4, Gurgaon",₹6.6 Cr,"₹14,667 /sqft",N80213331,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,2 Balconies,Study Room,2 Floors,South,10+ Year Old,4500 sqft (418.06 sqm) Plot Area,"['5 Fan', '10 Light', '1 Chimney', '1 Modular ...","['Park', 'Rain Water Harvesting']","['Palam Vihar Vyapar kendra', 'Palam vihar rai..."
1,"6 Bedroom House in DLF Phase 1, Gurgaon",https://www.99acres.com/6-bhk-bedroom-independ...,North East & Park Facing Corner 18 meter wide ...,₹25 Cr,"₹22,727 /sqft",Q84155382,https://www.99acres.com/6-bhk-bedroom-independ...,6 Bedrooms,10 Bathrooms,3+ Balconies,"Pooja Room,Study Room,Servant Room,Store Room",4 Floors,North-East,1 to 5 Year Old,11000 sqft (1021.93 sqm) Plot Area,"['1 Water Purifier', '13 Fan', '1 Exhaust Fan'...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...","['Anahat Hospital', 'Sikandarpur rmrg metro st..."
2,"2 Bedroom House in Sector 15 Part 2, Gurgaon",https://www.99acres.com/2-bhk-bedroom-independ...,"Sector 15 Part 2, Gurgaon",₹18.99 Cr,"₹42,031 /sqft",S82406206,https://www.99acres.com/2-bhk-bedroom-independ...,2 Bedrooms,2 Bathrooms,No Balcony,NaN,NaN,NaN,NaN,4518 sqft (419.74 sqm) Plot Area,[],NaN,"['govt sec school', 'Ahmed Hospital Multi Spec..."
3,"3 Bedroom House in Jhajjar Road, Gurgaon",https://www.99acres.com/3-bhk-bedroom-independ...,"Green Heritage Villas, Jhajjar",₹3.39 Cr,"₹3,766 /sqft",J84873064,https://www.99acres.com/3-bhk-bedroom-independ...,3 Bedrooms,4 Bathrooms,3 Balconies,"Study Room,Servant Room,Store Room",2 Floors,South,1 to 5 Year Old,9000 sqft (836.13 sqm) Plot Area,"['1 Water Purifier', '10 Fan', '1 Fridge', '1 ...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...",[]
4,"5 Bedroom House in Sector 109, Gurgaon",https://www.99acres.com/5-bhk-bedroom-independ...,Sobha International City Phase 3,₹14 Cr,"₹38,889 /sqft",Q83769064,https://www.99acres.com/5-bhk-bedroom-independ...,5 Bedrooms,6 Bathrooms,3+ Balconies,"Pooja Room,Study Room,Servant Room",2 Floors,East,1 to 5 Year Old,3600 sqft (334.45 sqm) Plot Area,NaN,"['Feng Shui / Vaastu Compliant', 'Private Gard...","['Palam Vihar Vyapar kendra', 'Palam vihar rai..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2263,"3 Bedroom House in Sector 49, Gurgaon",https://www.99acres.com/3-bhk-bedroom-independ...,Eros Rosewood City,₹4.5 Cr,"₹32,680 /sqft",V81604971,https://www.99acres.com/3-bhk-bedroom-independ...,3 Bedrooms,3 Bathrooms,1 Balcony,NaN,2 Floors,NaN,5 to 10 Year Old,1377 sqft (127.93 sqm) Plot Area,[],NaN,"['Silver Oak Universal', 'Leelavati Hospital',..."
2264,"4 Bedroom House in Pratap Nagar, Gurgaon",https://www.99acres.com/4-bhk-bedroom-independ...,383/13 Partap nagar,₹1.1 Cr,"₹6,077 /sqft",Q81598269,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,2 Bathrooms,No Balcony,NaN,2 Floors,East,Under Construction,1810 sqft (168.15 sqm) Plot Area,NaN,"['Water Storage', 'Park']",[]
2265,"4 Bedroom House in DLF Phase 3, Gurgaon",https://www.99acres.com/4-bhk-bedroom-independ...,National Media Centre,₹7 Cr,"₹32,407 /sqft",Q81594211,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,2 Balconies,"Study Room,Servant Room,Store Room,Pooja Room",2 Floors,NaN,10+ Year Old,2160 sqft (200.67 sqm) Plot Area,"['3 Wardrobe', '6 Fan', '1 Fridge', '4 Geyser'...","['Maintenance Staff', 'Water Storage', 'Visito...","['Deutsche bank', 'Dental Cure and Care Centre..."
2266,"2 Bedroom House in Devilal Colony, Gurgaon",https://www.99acres.com/2-bhk-bedroom-independ...,"Devilal Colony, Gurgaon",₹70 Lac,"₹76,923 /sqft",D81585663,https://www.99acres.com/2-bhk-bedroom-independ...,2 Bedrooms,1 Bathroom,1 Balcony,NaN,1 Floors,West,10+ Year Old,91 sqft (8.45 sqm) P

In [32]:
indepedent_house = indepedent_house_merge.drop(columns = ['link_x']).rename(columns ={'link_y':'link'})
indepedent_house

,property_name,society,price,area,property_id,link,bedrooms,bathrooms,balcony,additional_room,floor_info,facing,property_age,areawithtype,furnishing_details,features,nearby_location
0,"4 Bedroom House in Sector 4, Gurgaon","Sector 4, Gurgaon",₹6.6 Cr,"₹14,667 /sqft",N80213331,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,2 Balconies,Study Room,2 Floors,South,10+ Year Old,4500 sqft (418.06 sqm) Plot Area,"['5 Fan', '10 Light', '1 Chimney', '1 Modular ...","['Park', 'Rain Water Harvesting']","['Palam Vihar Vyapar kendra', 'Palam vihar rai..."
1,"6 Bedroom House in DLF Phase 1, Gurgaon",North East & Park Facing Corner 18 meter wide ...,₹25 Cr,"₹22,727 /sqft",Q84155382,https://www.99acres.com/6-bhk-bedroom-independ...,6 Bedrooms,10 Bathrooms,3+ Balconies,"Pooja Room,Study Room,Servant Room,Store Room",4 Floors,North-East,1 to 5 Year Old,11000 sqft (1021.93 sqm) Plot Area,"['1 Water Purifier', '13 Fan', '1 Exhaust Fan'...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...","['Anahat Hospital', 'Sikandarpur rmrg metro st..."
2,"2 Bedroom House in Sector 15 Part 2, Gurgaon","Sector 15 Part 2, Gurgaon",₹18.99 Cr,"₹42,031 /sqft",S82406206,https://www.99acres.com/2-bhk-bedroom-independ...,2 Bedrooms,2 Bathrooms,No Balcony,NaN,NaN,NaN,NaN,4518 sqft (419.74 sqm) Plot Area,[],NaN,"['govt sec school', 'Ahmed Hospital Multi Spec..."
3,"3 Bedroom House in Jhajjar Road, Gurgaon","Green Heritage Villas, Jhajjar",₹3.39 Cr,"₹3,766 /sqft",J84873064,https://www.99acres.com/3-bhk-bedroom-independ...,3 Bedrooms,4 Bathrooms,3 Balconies,"Study Room,Servant Room,Store Room",2 Floors,South,1 to 5 Year Old,9000 sqft (836.13 sqm) Plot Area,"['1 Water Purifier', '10 Fan', '1 Fridge', '1 ...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...",[]
4,"5 Bedroom House in Sector 109, Gurgaon",Sobha International City Phase 3,₹14 Cr,"₹38,889 /sqft",Q83769064,https://www.99acres.com/5-bhk-bedroom-independ...,5 Bedrooms,6 Bathrooms,3+ Balconies,"Pooja Room,Study Room,Servant Room",2 Floors,East,1 to 5 Year Old,3600 sqft (334.45 sqm) Plot Area,NaN,"['Feng Shui / Vaastu Compliant', 'Private Gard...","['Palam Vihar Vyapar kendra', 'Palam vihar rai..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2263,"3 Bedroom House in Sector 49, Gurgaon",Eros Rosewood City,₹4.5 Cr,"₹32,680 /sqft",V81604971,https://www.99acres.com/3-bhk-bedroom-independ...,3 Bedrooms,3 Bathrooms,1 Balcony,NaN,2 Floors,NaN,5 to 10 Year Old,1377 sqft (127.93 sqm) Plot Area,[],NaN,"['Silver Oak Universal', 'Leelavati Hospital',..."
2264,"4 Bedroom House in Pratap Nagar, Gurgaon",383/13 Partap nagar,₹1.1 Cr,"₹6,077 /sqft",Q81598269,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,2 Bathrooms,No Balcony,NaN,2 Floors,East,Under Construction,1810 sqft (168.15 sqm) Plot Area,NaN,"['Water Storage', 'Park']",[]
2265,"4 Bedroom House in DLF Phase 3, Gurgaon",National Media Centre,₹7 Cr,"₹32,407 /sqft",Q81594211,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,2 Balconies,"Study Room,Servant Room,Store Room,Pooja Room",2 Floors,NaN,10+ Year Old,2160 sqft (200.67 sqm) Plot Area,"['3 Wardrobe', '6 Fan', '1 Fridge', '4 Geyser'...","['Maintenance Staff', 'Water Storage', 'Visito...","['Deutsche bank', 'Dental Cure and Care Centre..."
2266,"2 Bedroom House in Devilal Colony, Gurgaon","Devilal Colony, Gurgaon",₹70 Lac,"₹76,923 /sqft",D81585663,https://www.99acres.com/2-bhk-bedroom-independ...,2 Bedrooms,1 Bathroom,1 Balcony,NaN,1 Floors,West,10+ Year Old,91 sqft (8.45 sqm) Plot Area,NaN,NaN,"['Prateek Nursing Home And Polyclinic', 'Shubh..."


In [34]:
indepedent_house.columns = indepedent_house.columns.str.lower()

In [35]:
indepedent_house.head()

,property_name,society,price,area,property_id,link,bedrooms,bathrooms,balcony,additional_room,floor_info,facing,property_age,areawithtype,furnishing_details,features,nearby_location
0,"4 Bedroom House in Sector 4, Gurgaon","Sector 4, Gurgaon",₹6.6 Cr,"₹14,667 /sqft",N80213331,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,2 Balconies,Study Room,2 Floors,South,10+ Year Old,4500 sqft (418.06 sqm) Plot Area,"['5 Fan', '10 Light', '1 Chimney', '1 Modular ...","['Park', 'Rain Water Harvesting']","['Palam Vihar Vyapar kendra', 'Palam vihar rai..."
1,"6 Bedroom House in DLF Phase 1, Gurgaon",North East & Park Facing Corner 18 meter wide ...,₹25 Cr,"₹22,727 /sqft",Q84155382,https://www.99acres.com/6-bhk-bedroom-independ...,6 Bedrooms,10 Bathrooms,3+ Balconies,"Pooja Room,Study Room,Servant Room,Store Room",4 Floors,North-East,1 to 5 Year Old,11000 sqft (1021.93 sqm) Plot Area,"['1 Water Purifier', '13 Fan', '1 Exhaust Fan'...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...","['Anahat Hospital', 'Sikandarpur rmrg metro st..."
2,"2 Bedroom House in Sector 15 Part 2, Gurgaon","Sector 15 Part 2, Gurgaon",₹18.99 Cr,"₹42,031 /sqft",S82406206,https://www.99acres.com/2-bhk-bedroom-independ...,2 Bedrooms,2 Bathrooms,No Balcony,NaN,NaN,NaN,NaN,4518 sqft (419.74 sqm) Plot Area,[],NaN,"['govt sec school', 'Ahmed Hospital Multi Spec..."
3,"3 Bedroom House in Jhajjar Road, Gurgaon","Green Heritage Villas, Jhajjar",₹3.39 Cr,"₹3,766 /sqft",J84873064,https://www.99acres.com/3-bhk-bedroom-independ...,3 Bedrooms,4 Bathrooms,3 Balconies,"Study Room,Servant Room,Store Room",2 Floors,South,1 to 5 Year Old,9000 sqft (836.13 sqm) Plot Area,"['1 Water Purifier', '10 Fan', '1 Fridge', '1 ...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...",[]
4,"5 Bedroom House in Sector 109, Gurgaon",Sobha International City Phase 3,₹14 Cr,"₹38,889 /sqft",Q83769064,https://www.99acres.com/5-bhk-bedroom-independ...,5 Bedrooms,6 Bathrooms,3+ Balconies,"Pooja Room,Study Room,Servant Room",2 Floors,East,1 to 5 Year Old,3600 sqft (334.45 sqm) Plot Area,NaN,"['Feng Shui / Vaastu Compliant', 'Private Gard...","['Palam Vihar Vyapar kendra', 'Palam vihar rai..."


In [ ]:
indepedent_house.to_csv("../../data/web_scraping/independent_house.csv", index= False)

In [37]:
indepedent_house.head()

,property_name,society,price,area,property_id,link,bedrooms,bathrooms,balcony,additional_room,floor_info,facing,property_age,areawithtype,furnishing_details,features,nearby_location
0,"4 Bedroom House in Sector 4, Gurgaon","Sector 4, Gurgaon",₹6.6 Cr,"₹14,667 /sqft",N80213331,https://www.99acres.com/4-bhk-bedroom-independ...,4 Bedrooms,4 Bathrooms,2 Balconies,Study Room,2 Floors,South,10+ Year Old,4500 sqft (418.06 sqm) Plot Area,"['5 Fan', '10 Light', '1 Chimney', '1 Modular ...","['Park', 'Rain Water Harvesting']","['Palam Vihar Vyapar kendra', 'Palam vihar rai..."
1,"6 Bedroom House in DLF Phase 1, Gurgaon",North East & Park Facing Corner 18 meter wide ...,₹25 Cr,"₹22,727 /sqft",Q84155382,https://www.99acres.com/6-bhk-bedroom-independ...,6 Bedrooms,10 Bathrooms,3+ Balconies,"Pooja Room,Study Room,Servant Room,Store Room",4 Floors,North-East,1 to 5 Year Old,11000 sqft (1021.93 sqm) Plot Area,"['1 Water Purifier', '13 Fan', '1 Exhaust Fan'...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...","['Anahat Hospital', 'Sikandarpur rmrg metro st..."
2,"2 Bedroom House in Sector 15 Part 2, Gurgaon","Sector 15 Part 2, Gurgaon",₹18.99 Cr,"₹42,031 /sqft",S82406206,https://www.99acres.com/2-bhk-bedroom-independ...,2 Bedrooms,2 Bathrooms,No Balcony,NaN,NaN,NaN,NaN,4518 sqft (419.74 sqm) Plot Area,[],NaN,"['govt sec school', 'Ahmed Hospital Multi Spec..."
3,"3 Bedroom House in Jhajjar Road, Gurgaon","Green Heritage Villas, Jhajjar",₹3.39 Cr,"₹3,766 /sqft",J84873064,https://www.99acres.com/3-bhk-bedroom-independ...,3 Bedrooms,4 Bathrooms,3 Balconies,"Study Room,Servant Room,Store Room",2 Floors,South,1 to 5 Year Old,9000 sqft (836.13 sqm) Plot Area,"['1 Water Purifier', '10 Fan', '1 Fridge', '1 ...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...",[]
4,"5 Bedroom House in Sector 109, Gurgaon",Sobha International City Phase 3,₹14 Cr,"₹38,889 /sqft",Q83769064,https://www.99acres.com/5-bhk-bedroom-independ...,5 Bedrooms,6 Bathrooms,3+ Balconies,"Pooja Room,Study Room,Servant Room",2 Floors,East,1 to 5 Year Old,3600 sqft (334.45 sqm) Plot Area,NaN,"['Feng Shui / Vaastu Compliant', 'Private Gard...","['Palam Vihar Vyapar kendra', 'Palam vihar rai..."
